In [23]:
# ================================
# SIMPLE SOLMAR EYEWEAR DASHBOARD
# ================================

!pip -q install plotly openpyxl

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import files
from IPython.display import display, HTML

# Upload Excel
uploaded = files.upload()
file = list(uploaded.keys())[0]

df = pd.read_excel(file)

# -------------------------------
# CLEAN DATA
# -------------------------------

df["date"] = pd.to_datetime(df["date"], errors="coerce")

spend_cols = [
    c for c in df.columns
    if c.startswith("spend_") or c == "Tiktok_DTC_Spends"
]

for c in ["total_revenue", "new_customer_revenue",
          "repeat_customer_revenue"] + spend_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

# -------------------------------
# KPIs
# -------------------------------

total_revenue = df["total_revenue"].sum()
new_revenue = df["new_customer_revenue"].sum()
repeat_revenue = df["repeat_customer_revenue"].sum()
marketing_spend = df[spend_cols].sum().sum()

roas = total_revenue / marketing_spend if marketing_spend else 0

# -------------------------------
# MONTHLY DATA
# -------------------------------

df["Month"] = df["date"].dt.to_period("M").astype(str)

monthly = df.groupby("Month").agg(
    Revenue=("total_revenue", "sum"),
    New=("new_customer_revenue", "sum"),
    Repeat=("repeat_customer_revenue", "sum")
).reset_index()

monthly_spend = (
    df.groupby("Month")[spend_cols]
    .sum()
    .sum(axis=1)
    .reset_index(name="Spend")
)

monthly = monthly.merge(monthly_spend, on="Month")

# -------------------------------
# CHANNEL DATA
# -------------------------------

channel_names = {
    "spend_Meta_Asc_Retargeting": "Meta ASC Retargeting",
    "spend_Meta_Asc": "Meta ASC",
    "spend_Meta_Retargeting_Manual": "Meta Retargeting",
    "spend_Meta_Broad_Manual": "Meta Broad",
    "spend_Meta_Non_Sales": "Meta Non-Sales",
    "spend_google_PMAX_Exist": "Google PMAX Existing",
    "spend_google_Search_Non_Brand": "Google Search Non-Brand",
    "spend_google_PMAX_New": "Google PMAX New",
    "spend_google_Search_Brand": "Google Search Brand",
    "spend_google_Shopping": "Google Shopping",
    "spend_snapchat": "Snapchat",
    "spend_Youtube": "YouTube",
    "Tiktok_DTC_Spends": "TikTok"
}

channels = pd.DataFrame({
    "Channel": [channel_names.get(c, c) for c in spend_cols],
    "Spend": [df[c].sum() for c in spend_cols]
})

channels = channels.sort_values("Spend")

# ==================================================
# DASHBOARD TITLE
# ==================================================

display(HTML("""
<div style="
font-family:Arial;
padding:15px 5px 10px 5px;
border-bottom:2px solid #333;
margin-bottom:15px;
">
<h1 style="margin:0;font-size:30px;color:#222;">
SOLMAR EYEWEAR
</h1>

<p style="margin:5px 0;color:#666;font-size:15px;">
Marketing Performance Dashboard
</p>
</div>
"""))

# ==================================================
# KPI CARDS
# ==================================================

display(HTML(f"""
<div style="
display:grid;
grid-template-columns:repeat(4,1fr);
gap:12px;
font-family:Arial;
margin-bottom:20px;
">

<div style="
background:#f5f5f5;
padding:15px;
border:1px solid #ddd;
border-radius:8px;
">
<div style="font-size:12px;color:#777;">TOTAL REVENUE</div>
<div style="font-size:23px;font-weight:bold;color:#222;">
₹{total_revenue:,.0f}
</div>
</div>

<div style="
background:#f5f5f5;
padding:15px;
border:1px solid #ddd;
border-radius:8px;
">
<div style="font-size:12px;color:#777;">MARKETING SPEND</div>
<div style="font-size:23px;font-weight:bold;color:#222;">
₹{marketing_spend:,.0f}
</div>
</div>

<div style="
background:#f5f5f5;
padding:15px;
border:1px solid #ddd;
border-radius:8px;
">
<div style="font-size:12px;color:#777;">BLENDED ROAS</div>
<div style="font-size:23px;font-weight:bold;color:#222;">
{roas:.2f}×
</div>
</div>

<div style="
background:#f5f5f5;
padding:15px;
border:1px solid #ddd;
border-radius:8px;
">
<div style="font-size:12px;color:#777;">NEW CUSTOMER REVENUE</div>
<div style="font-size:23px;font-weight:bold;color:#222;">
₹{new_revenue:,.0f}
</div>
</div>

</div>
"""))

# ==================================================
# CREATE 3 SIMPLE GRAPHS
# ==================================================

fig = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{"colspan":2}, None],
        [{}, {}]
    ],
    subplot_titles=[
        "Revenue and Marketing Spend",
        "Marketing Spend by Channel",
        "New vs Repeat Customer Revenue"
    ],
    vertical_spacing=0.18,
    horizontal_spacing=0.12
)

# -------------------------------
# GRAPH 1
# -------------------------------

fig.add_trace(
    go.Scatter(
        x=monthly["Month"],
        y=monthly["Revenue"],
        mode="lines+markers",
        name="Revenue",
        line=dict(width=3)
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=monthly["Month"],
        y=monthly["Spend"],
        mode="lines+markers",
        name="Marketing Spend",
        line=dict(width=3)
    ),
    row=1,
    col=1
)

# -------------------------------
# GRAPH 2
# -------------------------------

fig.add_trace(
    go.Bar(
        x=channels["Spend"],
        y=channels["Channel"],
        orientation="h",
        name="Spend",
        marker_color="#555555",
        showlegend=False
    ),
    row=2,
    col=1
)

# -------------------------------
# GRAPH 3
# -------------------------------

fig.add_trace(
    go.Bar(
        x=["New Customers", "Repeat Customers"],
        y=[new_revenue, repeat_revenue],
        marker_color="#777777",
        showlegend=False
    ),
    row=2,
    col=2
)

# ==================================================
# SIMPLE STYLE
# ==================================================

fig.update_layout(
    height=850,
    paper_bgcolor="white",
    plot_bgcolor="white",

    font=dict(
        family="Arial",
        color="#333333"
    ),

    title=dict(
        text="Marketing Overview",
        x=0.5,
        font=dict(size=20)
    ),

    margin=dict(
        l=50,
        r=80,
        t=90,
        b=50
    ),

    legend=dict(
        orientation="h",
        y=1.02,
        x=0.5,
        xanchor="center"
    )
)

fig.update_xaxes(
    showgrid=True,
    gridcolor="#eeeeee"
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="#eeeeee"
)

fig.show()

# ==================================================
# SMALL FOOTNOTE
# ==================================================

display(HTML("""
<div style="
font-family:Arial;
font-size:12px;
color:#777;
margin-top:8px;
padding:10px;
border-top:1px solid #ddd;
">
<b>Note:</b> Blended ROAS = Total Revenue ÷ Total Marketing Spend.
Channel-level ROAS is not available because the dataset does not contain
revenue attributed separately to each marketing channel.
</div>
"""))


Saving lifesight.xlsx to lifesight (8).xlsx
